# 07 — Scaling Benchmark: Serial vs MP vs Dask

Compares wall-clock time across 7 configurations on a 1M-sample bank:

| Config | Module | n_workers | OMP threads (main/workers) |
|---|---|---|---|
| Serial | `inference` | — | 8 / — |
| MP 8w | `mp_inference` | 8 | 1 / 1 |
| MP 16w | `mp_inference` | 16 | 1 / 1 |
| MP 32w | `mp_inference` | 32 | 1 / 1 |
| Dask 8w | `dask_inference` | 8 | 1 / 1 |
| Dask 16w | `dask_inference` | 16 | 1 / 1 |
| Dask 32w | `dask_inference` | 32 | 1 / 1 |

Each config runs in **a subprocess** with its own environment variables so thread
counts are cleanly isolated.  Serial gets `OMP_NUM_THREADS=8` so numpy/BLAS can
use 8 cores; MP/Dask workers each get `OMP_NUM_THREADS=1` (controlled per process).

## Cell 1 — Imports & constants

In [ ]:
import io
import re
import subprocess
import sys
import textwrap
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────
REPO_ROOT   = Path("../..").resolve()
EVENT_PATH  = REPO_ROOT / "gpu/artifacts/nb04/nb04_event.npz"
BANK_FOLDER = REPO_ROOT / "gpu/artifacts/profile_run/test_bank_1048576"
OUTPUT_ROOT = REPO_ROOT / "gpu/artifacts/benchmark_scaling"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

assert EVENT_PATH.exists(),  f"Event not found: {EVENT_PATH}"
assert BANK_FOLDER.exists(), f"Bank not found: {BANK_FOLDER}"

# ── BLAS thread count for serial baseline ─────────────────────────────
SERIAL_N_THREADS = 8   # numpy/BLAS threads for inference.run()
MP_DASK_N_THREADS = 1  # per-worker threads for mp/dask (each worker is 1 process)

# ── Run kwargs shared across all configs ───────────────────────────────
RUN_KWARGS = dict(
    n_ext=512, n_phi=32, n_t=64,
    blocksize=2048, single_detector_blocksize=2048,
    mchirp_guess=None, seed=42,
    max_incoherent_lnlike_drop=20.0,
)

# Stage labels for tables/plots
STAGE_KEYS = ["1_setup", "2_incoherent", "3_crossbank", "4_extrinsic", "5_coherent", "6_postprocess"]
STAGE_LABELS = {
    "1_setup":       "Stage 1\n(setup)",
    "2_incoherent":  "Stage 2\n(incoherent)",
    "3_crossbank":   "Stage 3\n(cross-bank)",
    "4_extrinsic":   "Stage 4\n(extrinsic)",
    "5_coherent":    "Stage 5\n(coherent)",
    "6_postprocess": "Stage 6\n(postprocess)",
}

print(f"Event:           {EVENT_PATH}")
print(f"Bank:            {BANK_FOLDER}")
print(f"Output:          {OUTPUT_ROOT}")
print(f"Serial threads:  {SERIAL_N_THREADS}")
print(f"MP/Dask threads: {MP_DASK_N_THREADS} per worker")

## Cell 2 — `run_timed()` helper

Spawns each benchmark in a **subprocess** with its own `OMP_NUM_THREADS` so thread
counts are fully isolated.  Captures stdout to parse per-stage timings printed by
`mp_inference` / `dask_inference`; for serial a wrapper measures each step.

In [ ]:
# Stage timing line pattern emitted by mp_inference and dask_inference:
#   "  Stage 1 (setup)             12.3 s"
_STAGE_RE = re.compile(r"Stage (\d)\s+\((\w+)\)\s+([\d.]+)\s+s")
_TOTAL_RE = re.compile(r"^__TOTAL__:([\d.]+)", re.MULTILINE)


def _parse_timings(text: str) -> dict:
    """Extract per-stage timings and __TOTAL__ from subprocess stdout."""
    result = {}
    for m in _STAGE_RE.finditer(text):
        key = f"{m.group(1)}_{m.group(2)}"
        result[key] = float(m.group(3))
    m = _TOTAL_RE.search(text)
    if m:
        result["total"] = float(m.group(1))
    return result


# Python runner script template — filled in per call
_RUNNER_TEMPLATE = textwrap.dedent("""\
    import os
    os.environ["OMP_NUM_THREADS"]      = "{n_threads}"
    os.environ["MKL_NUM_THREADS"]      = "{n_threads}"
    os.environ["OPENBLAS_NUM_THREADS"] = "{n_threads}"
    os.environ["NUMEXPR_NUM_THREADS"]  = "{n_threads}"

    import time, warnings
    warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
    from dot_pe import {module_name} as _mod

    _kwargs = dict(
        event={event!r},
        bank_folder={bank!r},
        rundir={rundir!r},
        n_ext={n_ext}, n_phi={n_phi}, n_t={n_t},
        blocksize={blocksize},
        single_detector_blocksize={single_detector_blocksize},
        mchirp_guess={mchirp_guess!r},
        seed={seed!r},
        max_incoherent_lnlike_drop={max_incoherent_lnlike_drop},
        {extra_kwargs}
    )

    _t0 = time.perf_counter()
    _mod.run(**_kwargs)
    _total = time.perf_counter() - _t0
    print(f"__TOTAL__:{{_total:.3f}}")
""")


def run_timed(
    module_name: str,
    rundir_name: str,
    n_threads: int = MP_DASK_N_THREADS,
    **extra_kwargs,
) -> dict:
    """
    Run ``module_name.run()`` in a subprocess with ``n_threads`` OMP threads.
    Captures stdout to parse per-stage timings; echos output to cell.
    Returns ``{stage_key: seconds, ..., 'total': seconds}``.
    """
    rundir = str(OUTPUT_ROOT / rundir_name)

    extra_str = ", ".join(
        f"{k}={v!r}" for k, v in extra_kwargs.items()
    )

    script = _RUNNER_TEMPLATE.format(
        n_threads=n_threads,
        module_name=module_name,
        event=str(EVENT_PATH),
        bank=str(BANK_FOLDER),
        rundir=rundir,
        extra_kwargs=extra_str,
        **RUN_KWARGS,
    )

    proc = subprocess.run(
        [sys.executable, "-c", script],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(proc.stdout)  # echo to notebook cell

    if proc.returncode != 0:
        raise RuntimeError(
            f"{module_name} subprocess failed (exit {proc.returncode})"
        )

    return _parse_timings(proc.stdout)

## Cell 3 — Run all 7 experiments

> Each run is fully isolated in its own subprocess.  Running them sequentially is
> intentional — parallel benchmarks would compete for CPU resources.

In [ ]:
CONFIGS = [
    # (label,      module_name,       extra_kwargs,          rundir_name,         n_threads)
    ("Serial",    "inference",        {},                    "benchmark_serial",   SERIAL_N_THREADS),
    ("MP 8w",     "mp_inference",     {"n_workers":  8},     "benchmark_mp_8w",    MP_DASK_N_THREADS),
    ("MP 16w",    "mp_inference",     {"n_workers": 16},     "benchmark_mp_16w",   MP_DASK_N_THREADS),
    ("MP 32w",    "mp_inference",     {"n_workers": 32},     "benchmark_mp_32w",   MP_DASK_N_THREADS),
    ("Dask 8w",   "dask_inference",   {"n_workers":  8},     "benchmark_dask_8w",  MP_DASK_N_THREADS),
    ("Dask 16w",  "dask_inference",   {"n_workers": 16},     "benchmark_dask_16w", MP_DASK_N_THREADS),
    ("Dask 32w",  "dask_inference",   {"n_workers": 32},     "benchmark_dask_32w", MP_DASK_N_THREADS),
]

results = {}  # label -> stage_timings dict

for label, module_name, extra, rundir_name, n_threads in CONFIGS:
    print(f"\n{'='*60}")
    print(f"Running: {label}  (OMP_NUM_THREADS={n_threads})  →  {rundir_name}")
    print(f"{'='*60}")
    timings = run_timed(module_name, rundir_name, n_threads=n_threads, **extra)
    results[label] = timings
    print(f">>> {label} total: {timings.get('total', float('nan')):.1f} s")

## Cell 4 — Results table

In [ ]:
all_cols = STAGE_KEYS + ["total"]
df_results = pd.DataFrame(
    {
        label: {col: results[label].get(col, float("nan")) for col in all_cols}
        for label in results
    }
).T
df_results.index.name = "Config"

display_cols = {
    "1_setup":       "S1 setup",
    "2_incoherent":  "S2 incoherent",
    "3_crossbank":   "S3 cross-bank",
    "4_extrinsic":   "S4 extrinsic",
    "5_coherent":    "S5 coherent",
    "6_postprocess": "S6 postproc",
    "total":         "Total (s)",
}
df_display = df_results.rename(columns=display_cols)
print(f"Wall-clock time per stage (s).  Serial uses {SERIAL_N_THREADS} BLAS threads; MP/Dask workers use {MP_DASK_N_THREADS}.")
df_display.round(1)

## Cell 5 — Save results

In [ ]:
csv_path = OUTPUT_ROOT / "benchmark_results.csv"
df_results.to_csv(csv_path)
print(f"Saved: {csv_path}")

## Cell 6 — Grouped bar chart (per-stage wall-clock)

In [ ]:
config_names = list(results.keys())
n_configs = len(config_names)
x = np.arange(len(STAGE_KEYS))
width = 0.8 / n_configs

fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.tab10(np.linspace(0, 0.9, n_configs))

for i, cfg in enumerate(config_names):
    vals = [results[cfg].get(k, 0.0) for k in STAGE_KEYS]
    offset = (i - n_configs / 2 + 0.5) * width
    ax.bar(x + offset, vals, width, label=cfg, color=colors[i])

ax.set_xticks(x)
ax.set_xticklabels([STAGE_LABELS[k] for k in STAGE_KEYS], fontsize=10)
ax.set_ylabel("Wall-clock time (s)")
ax.set_title(
    f"Scaling Benchmark: per-stage wall-clock time\n"
    f"(1M-sample bank, IMRPhenomXODE; serial={SERIAL_N_THREADS} BLAS threads, MP/Dask={MP_DASK_N_THREADS} thread/worker)"
)
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig_path = OUTPUT_ROOT / "benchmark_stages.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")

## Cell 7 — Total wall-clock bar chart

In [ ]:
totals = [results[cfg].get("total", float("nan")) for cfg in config_names]

fig2, ax2 = plt.subplots(figsize=(9, 4))
colors = plt.cm.tab10(np.linspace(0, 0.9, n_configs))
bars = ax2.bar(config_names, totals, color=colors)
for bar, val in zip(bars, totals):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(totals) * 0.01,
        f"{val:.0f}s",
        ha="center", va="bottom", fontsize=9,
    )
ax2.set_ylabel("Total wall-clock time (s)")
ax2.set_title("Scaling Benchmark: total wall-clock time")
ax2.set_ylim(bottom=0, top=max(totals) * 1.12)
ax2.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig2_path = OUTPUT_ROOT / "benchmark_total.png"
fig2.savefig(fig2_path, dpi=150)
plt.show()
print(f"Saved: {fig2_path}")

## Cell 8 — Speedup table (relative to Serial)

In [ ]:
serial_timings = results["Serial"]

speedup_rows = {}
for cfg in config_names:
    row = {}
    for k in STAGE_KEYS + ["total"]:
        s = serial_timings.get(k, float("nan"))
        c = results[cfg].get(k, float("nan"))
        row[k] = s / c if (s and c and c > 0) else float("nan")
    speedup_rows[cfg] = row

df_speedup = pd.DataFrame(speedup_rows).T
df_speedup.index.name = "Config"
df_speedup = df_speedup.rename(columns=display_cols)

print(f"Speedup vs Serial ({SERIAL_N_THREADS} BLAS threads):")
df_speedup.round(2)

## Cell 9 — Per-stage speedup chart (parallel stages 2, 4, 5)

In [ ]:
parallel_stages = ["2_incoherent", "4_extrinsic", "5_coherent"]
parallel_labels = [STAGE_LABELS[k].replace("\n", " ") for k in parallel_stages]

parallel_configs = [cfg for cfg in config_names if cfg != "Serial"]
n_p = len(parallel_configs)
x3 = np.arange(len(parallel_stages))
w3 = 0.8 / n_p
colors_p = plt.cm.tab10(np.linspace(0, 0.9, n_configs))[1:]  # skip Serial

fig3, ax3 = plt.subplots(figsize=(9, 4))
for i, cfg in enumerate(parallel_configs):
    vals = [speedup_rows[cfg].get(k, float("nan")) for k in parallel_stages]
    offset = (i - n_p / 2 + 0.5) * w3
    ax3.bar(x3 + offset, vals, w3, label=cfg, color=colors_p[i])

ax3.axhline(1.0, color="grey", linestyle="--", linewidth=0.8, label="Serial baseline")
ax3.set_xticks(x3)
ax3.set_xticklabels(parallel_labels)
ax3.set_ylabel("Speedup vs Serial")
ax3.set_title(
    f"Parallel-stage speedup (stages 2, 4, 5)\n"
    f"Serial baseline = {SERIAL_N_THREADS} BLAS threads"
)
ax3.legend(fontsize=9)
ax3.set_ylim(bottom=0)
ax3.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig3_path = OUTPUT_ROOT / "benchmark_speedup.png"
fig3.savefig(fig3_path, dpi=150)
plt.show()
print(f"Saved: {fig3_path}")